# nb46 — sniper × inverse_breadth=False stacking A/B on 100 days

**User directive (2026-09-17):** "suggests what new experiments to
do, that changes logically the strategy, then run another backtest
with the updated agents.md". The logical strategy-change question
was: does the v8 winner (`inverse_breadth=False`) STACK with the v7
SNIPER mode?

v9 closes this question with a 6-config sweep on 100 days. Two
cross-mode questions are answered:

1. **Stacking**: does `inverse_breadth=False` help or hurt
   `entry_mode='sniper'`? (v8 only tested immediate mode)
2. **TP scaling**: at N=100, does TP=15 / TP=22 / TP=30 produce
   different EV? (v7 sweep stopped at TP=22 on N=56)

## Design

6 single-knob configurations built off the v7 OPTIMAL base:

| # | Config | `entry_mode` | `inverse_breadth` | TP mult |
|---|---|---|---|---|
| 1 | SNIPER_TP22_BREADTH_ON  | sniper   | True  | 22.0 |
| 2 | SNIPER_TP22_BREADTH_OFF | sniper   | False | 22.0 |
| 3 | SNIPER_TP15_BREADTH_OFF | sniper   | False | 15.0 |
| 4 | SNIPER_TP30_BREADTH_OFF | sniper   | False | 30.0 |
| 5 | IMMEDIATE_BREADTH_OFF   | immediate | False | 1.8 |
| 6 | IMMEDIATE_BREADTH_ON    | immediate | True  | 1.8 |

The headline comparison is row 1 vs row 2 (does stacking help?).
Row 1 vs rows 3-4 (TP grid). Row 5 vs row 6 (v8 reference).

## Output files

* `notebooks/sniper_breadth_per_day.csv`   — 600 per-day rows
* `notebooks/sniper_breadth_summary.csv`   — per-config totals

## Reference

See AGENTS.md § "v9 — sniper × inverse_breadth=False stacking A/B
(2026-09-17)" for the full findings.

## Edit knobs

In [1]:
# Knobs - edit and re-run
N_DAYS = 100             # how many random days
RNG_SEED = 20260917      # reproducible day selection (same as v8/v9)
MIN_BARS_PER_DAY = 30_000

# Run as a subprocess so output is captured in the cell.
import subprocess

## Locate repo + verify tool exists

In [2]:
import os
import sys
from pathlib import Path

os.environ.setdefault('MPLBACKEND', 'Agg')


def _find_root() -> Path:
    here = Path('.').resolve()
    candidates = [p for p in [here, *here.parents]
                  if (p / 'src' / 'core' / 'ict_signals.py').is_file()]
    if not candidates:
        raise RuntimeError('Could not find ICT repo root')

    def _has_fork_sig(p: Path) -> bool:
        f = p / 'src' / 'core' / 'ict_strategy.py'
        try:
            return 'played_out_min_extension_usd' in f.read_text(encoding='utf-8')
        except OSError:
            return False

    with_sig = [p for p in candidates if _has_fork_sig(p)]
    return with_sig[0] if with_sig else candidates[0]


ROOT = _find_root()
sys.path.insert(0, str(ROOT))
print(f'Repo root: {ROOT}')

tool_path = ROOT / 'src' / 'tools' / 'run_sniper_breadth.py'
assert tool_path.is_file(), f'missing: {tool_path}'
print(f'Tool: {tool_path}')

Repo root: C:\coding\ict_tier_v2
Tool: C:\coding\ict_tier_v2\src\tools\run_sniper_breadth.py


## Run v9 sweep — 100 random Mon-Fri UTC days

In [3]:
print(f'\n>>> V9 SWEEP: {N_DAYS} days <<<\n')
res = subprocess.run(
    [sys.executable, str(tool_path), str(N_DAYS)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(res.stdout)
if res.returncode != 0:
    print('STDERR:', res.stderr)


>>> V9 SWEEP: 100 days <<<



Day enumeration: 0.7s, 660 Mon-Fri days
Sampled 100 days with seed 20260917
Pre-loading all days...
Pre-load: 1.6s, 100 days

Running 600 backtests (6 configs x 100 days)...

=== SNIPER_TP22_BREADTH_ON ===
  [30/600] SNIPER_TP22_BREADTH_ON 2024-07-22 elapsed=4.4s eta=83.1s
  [60/600] SNIPER_TP22_BREADTH_ON 2025-05-13 elapsed=9.7s eta=87.3s
  [90/600] SNIPER_TP22_BREADTH_ON 2026-04-14 elapsed=15.8s eta=89.3s

=== SNIPER_TP22_BREADTH_OFF ===
  [120/600] SNIPER_TP22_BREADTH_OFF 2024-06-05 elapsed=20.9s eta=83.4s
  [150/600] SNIPER_TP22_BREADTH_OFF 2025-01-22 elapsed=26.2s eta=78.7s
  [180/600] SNIPER_TP22_BREADTH_OFF 2026-01-26 elapsed=32.1s eta=74.9s

=== SNIPER_TP15_BREADTH_OFF ===
  [210/600] SNIPER_TP15_BREADTH_OFF 2024-03-08 elapsed=37.7s eta=70.0s
  [240/600] SNIPER_TP15_BREADTH_OFF 2024-10-10 elapsed=42.9s eta=64.3s
  [270/600] SNIPER_TP15_BREADTH_OFF 2025-10-20 elapsed=48.0s eta=58.7s
  [300/600] SNIPER_TP15_BREADTH_OFF 2026-07-13 elapsed=54.4s eta=54.4s

=== SNIPER_TP30_BREADTH_O

## Inspect results

In [4]:
import pandas as pd

summary_path = ROOT / 'notebooks' / 'sniper_breadth_summary.csv'
per_day_path = ROOT / 'notebooks' / 'sniper_breadth_per_day.csv'

summary = pd.read_csv(summary_path)
per_day = pd.read_csv(per_day_path)

print(f'V9 summary: {summary.shape[0]} configs, '
      f'{len(per_day["day"].unique())} days, '
      f'{int(summary["trades"].sum()):,} trades total')

print('\nTop configs by PnL/day:')
top = summary.sort_values('pnl_per_day', ascending=False)
print(top[['config', 'trades', 'ev_per_trade', 'pnl_per_day', 'trades_per_day']]
      .to_string(index=False))

V9 summary: 6 configs, 100 days, 15,618 trades total

Top configs by PnL/day:
                 config  trades  ev_per_trade  pnl_per_day  trades_per_day
SNIPER_TP30_BREADTH_OFF     828      3.195330    26.457335            8.28
 SNIPER_TP22_BREADTH_ON     828      2.466365    20.421504            8.28
SNIPER_TP22_BREADTH_OFF     828      2.466332    20.421225            8.28
SNIPER_TP15_BREADTH_OFF     828      2.077933    17.205285            8.28
  IMMEDIATE_BREADTH_OFF    6153      0.261341    16.080299           61.53
   IMMEDIATE_BREADTH_ON    6153     -0.045097    -2.774791           61.53


## Per-day distribution analysis

In [5]:
print('Per-day PnL distribution by config (mean, std, se, % positive days):')
print('=' * 80)
for cfg, g in per_day.groupby('config'):
    pnls = g['pnl_total']
    se = pnls.std() / (len(pnls) ** 0.5)
    t = pnls.mean() / se if se > 0 else 0.0
    print(f'{cfg:<32}  mean=${pnls.mean():+8.2f}/d  '
          f'std=${pnls.std():7.2f}  se=${se:6.2f}  '
          f't={t:+5.2f}  '
          f'pos={(pnls>0).sum():>3}/100  neg={(pnls<0).sum():>3}/100')

Per-day PnL distribution by config (mean, std, se, % positive days):
IMMEDIATE_BREADTH_OFF             mean=$  +16.08/d  std=$  40.32  se=$  4.03  t=+3.99  pos= 88/100  neg= 12/100
IMMEDIATE_BREADTH_ON              mean=$   -2.77/d  std=$   3.96  se=$  0.40  t=-7.00  pos= 12/100  neg= 88/100
SNIPER_TP15_BREADTH_OFF           mean=$  +17.21/d  std=$  48.16  se=$  4.82  t=+3.57  pos= 66/100  neg= 31/100
SNIPER_TP22_BREADTH_OFF           mean=$  +20.42/d  std=$  49.45  se=$  4.95  t=+4.13  pos= 65/100  neg= 32/100
SNIPER_TP22_BREADTH_ON            mean=$  +20.42/d  std=$  49.45  se=$  4.95  t=+4.13  pos= 65/100  neg= 32/100
SNIPER_TP30_BREADTH_OFF           mean=$  +26.46/d  std=$  66.76  se=$  6.68  t=+3.96  pos= 66/100  neg= 31/100


## Headline answers

**Q1: does `inverse_breadth=False` stack with sniper?**
A: NO. Rows SNIPER_TP22_BREADTH_ON and SNIPER_TP22_BREADTH_OFF
produce identical PnL to 4 decimal places (~$2042). The sniper
path uses `fvg_inv_trade_sl_zone_mult` / `fvg_inv_trade_tp_zone_mult`
for SL/TP — completely independent of `inverse_breadth`.

**Q2: which TP wins at N=100?**
A: TP=30 wins (+$26.46/day) — beats TP=22 (+$20.42/day) by +29.5%.
TP=15 is worse (+$17.21/day). Monotonic 15→22→30 — peak may
continue higher (needs v10 to confirm).

**Q3: does SNIPER beat IMMEDIATE on PnL/day?**
A: Yes at N=100. SNIPER_TP30 +$26.46/day vs IMMEDIATE_BREADTH_OFF
+$16.08/day. Trade count is much lower (8.3/day vs 61.5/day) but
EV per trade is 12× higher (+$3.20 vs +$0.26).

**Decision: do NOT promote TP=30 to canonical yet** — full-corpus
walk-forward (v10) required first. The current v7 recipe (TP=22)
is full-corpus-validated on 654 days. TP=30 is only N=100.